# Go2 Track 2 Bonus Project: Public Colab Template

This notebook runs the starter 200 m oval track bonus pipeline. It deliberately does not download a solved checkpoint or high-level planner.

Track 2 has three routes: proposal-based final project, Go2 oval-track tournament, or both for bonus. This notebook supports the tournament route and can also provide reusable code for a proposal-based project.

By the end, you should know how to:

- install the MuJoCo/Brax/Go2 dependencies
- inspect the low-level HW1-style locomotion environment
- train or reuse a low-level `best_checkpoint/`
- run the starter track evaluation
- optionally tune the starter high-level planner
- package the required submission artifacts


## 1. Configure repository URLs

If you fork this repository, change `COURSE_REPO_URL` to your own GitHub repo before running setup. Colab storage is temporary, so push your changes to GitHub often.


In [ ]:
from pathlib import Path
import io
import os
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.request
from urllib.parse import urlparse

COURSE_REPO_URL = "https://github.com/WeijieLai1024/Final-Project-Track-2-Bonus-Project.git"
COURSE_REPO_BRANCH = "main"
BASE_DIR = Path("/content") if Path("/content").exists() else Path("/tmp/go2_track_bonus_colab")
BASE_DIR.mkdir(parents=True, exist_ok=True)
COURSE_REPO_DIR = BASE_DIR / "go2_track_bonus_repo"

PLAYGROUND_REPO = "https://github.com/google-deepmind/mujoco_playground.git"
PLAYGROUND_REF = "dd38c285c6d54266287081e516109f0b15985818"

UNITREE_MUJOCO_REPO = "https://github.com/unitreerobotics/unitree_mujoco.git"
UNITREE_MUJOCO_REF = "1a37b051a10be723405b7ed6dc839361af036d88"

MENAGERIE_REPO = "https://github.com/deepmind/mujoco_menagerie.git"
MENAGERIE_REF = "1b86ece576591213e2b666ebf59508454200ca97"

PLAYGROUND_DIR = BASE_DIR / "mujoco_playground"
UNITREE_DIR = BASE_DIR / "unitree_mujoco"
MENAGERIE_DIR = PLAYGROUND_DIR / "mujoco_playground" / "external_deps" / "mujoco_menagerie"

def run(cmd):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True)

def github_archive_url(repo_url: str, ref: str) -> str:
    repo_path = urlparse(repo_url).path.strip("/")
    if repo_path.endswith(".git"):
        repo_path = repo_path[:-4]
    return f"https://codeload.github.com/{repo_path}/tar.gz/{ref}"

def download_repo_snapshot(repo_url: str, ref: str, target_dir: Path) -> None:
    archive_url = github_archive_url(repo_url, ref)
    print(f"+ download {archive_url}")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir = Path(tempfile.mkdtemp(prefix=f"{target_dir.name}_", dir=str(target_dir.parent)))
    try:
        with urllib.request.urlopen(archive_url) as response:
            payload = response.read()
        with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
            archive.extractall(tmp_dir)
        extracted_dirs = [path for path in tmp_dir.iterdir() if path.is_dir()]
        if len(extracted_dirs) != 1:
            raise RuntimeError(f"Expected one extracted directory, got {extracted_dirs}")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.move(str(extracted_dirs[0]), str(target_dir))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

def checkout_existing_repo(target_dir: Path, ref: str) -> None:
    try:
        run(["git", "-C", target_dir, "fetch", "--all", "--tags"])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git fetch failed for {target_dir}: {exc}. Trying local checkout.")
    run(["git", "-C", target_dir, "checkout", ref])

def ensure_pinned_repo(repo_url: str, ref: str, target_dir: Path) -> None:
    if target_dir.exists() and (target_dir / ".git").exists():
        try:
            checkout_existing_repo(target_dir, ref)
            return
        except subprocess.CalledProcessError as exc:
            print(f"[warn] local git checkout failed for {target_dir}: {exc}. Re-downloading snapshot.")
            shutil.rmtree(target_dir)
    elif target_dir.exists():
        shutil.rmtree(target_dir)

    try:
        run(["git", "clone", repo_url, target_dir])
        checkout_existing_repo(target_dir, ref)
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git path failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, ref, target_dir)

def ensure_course_repo(repo_url: str, branch: str, target_dir: Path) -> None:
    if target_dir.exists():
        return
    try:
        run(["git", "clone", repo_url, target_dir])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git clone failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, branch, target_dir)

if "google.colab" in sys.modules:
    print("Running inside Colab.")
else:
    print("This notebook was designed for Colab, but local execution may also work.")


## 2. Install system packages and clone repositories


In [ ]:
import shutil
if shutil.which("ffmpeg") is None:
    if Path("/content").exists():
        run(["apt-get", "update", "-qq"])
        run(["apt-get", "install", "-y", "ffmpeg"])
    else:
        print("[local] ffmpeg is not on PATH; imageio-ffmpeg from requirements is enough for local tests.")
!python -m pip install -q -U pip setuptools wheel
!python -m pip uninstall -y playground || true

ensure_pinned_repo(PLAYGROUND_REPO, PLAYGROUND_REF, PLAYGROUND_DIR)
ensure_pinned_repo(UNITREE_MUJOCO_REPO, UNITREE_MUJOCO_REF, UNITREE_DIR)
ensure_course_repo(COURSE_REPO_URL, COURSE_REPO_BRANCH, COURSE_REPO_DIR)
ensure_pinned_repo(MENAGERIE_REPO, MENAGERIE_REF, MENAGERIE_DIR)

!python -m pip install -q -r {COURSE_REPO_DIR / 'configs' / 'colab_requirements.txt'}
%cd {PLAYGROUND_DIR}
!python -m pip install -q -e .
%cd {COURSE_REPO_DIR}

import sys
if str(PLAYGROUND_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(PLAYGROUND_DIR.resolve()))

import jax
import mujoco_playground

print("JAX devices:", jax.devices())
print("JAX backend:", jax.default_backend())
print("mujoco_playground imported from:", mujoco_playground.__file__)
expected_playground = str(PLAYGROUND_DIR.resolve())
if expected_playground not in str(Path(mujoco_playground.__file__).resolve()):
    raise RuntimeError(f"Expected mujoco_playground to be imported from {expected_playground}")


## 3. Read the assignment requirements

These files define the course routes, goals, allowed changes, metrics, deliverables, report questions, grading distribution, tournament ranking, and high-level optimization scaffold.


In [ ]:
%cd {COURSE_REPO_DIR}
!sed -n '1,260p' docs/track2_assignment_handout.md
!sed -n '1,260p' docs/assignment_requirements.md
!sed -n '1,240p' docs/controller_interface.md
!sed -n '1,260p' docs/high_level_optimization_guide.md


## 4. Copy Go2 assets


In [ ]:
%cd {COURSE_REPO_DIR}
!python scripts/copy_go2_assets.py --unitree-dir {UNITREE_DIR} --course-dir {COURSE_REPO_DIR}


## 5. Inspect the low-level Go2 environment


In [ ]:
%cd {COURSE_REPO_DIR}
!python inspect_env.py --stage-name stage_2


## 6. Read the important starter files


In [ ]:
!sed -n '1,180p' go2_pg_env/joystick.py
!sed -n '1,220p' go2_pg_env/track.py
!sed -n '1,220p' track_bonus/controller_interface.py
!sed -n '1,220p' track_bonus/planner.py
!sed -n '1,220p' run_track_bonus.py


## 7. Define a Colab-friendly low-level training config

The default keeps the HW1-style low-level baseline. You can increase stage 2 turning/yaw training after you understand the starter.


In [ ]:
import json

runtime_config = {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [256, 256, 128],
    "value_hidden_layer_sizes": [256, 256, 128],
    "stage_1_num_timesteps": 10_000_000,
    "stage_2_num_timesteps": 5_000_000,
}

config_path = COURSE_REPO_DIR / "configs" / "colab_runtime_config.json"
base_config_path = COURSE_REPO_DIR / "configs" / "course_config.json"
base_config = json.loads(base_config_path.read_text())
base_config["runtime_overrides"] = runtime_config
config_path.write_text(json.dumps(base_config, indent=2))
print("wrote", config_path)


## 8. Dry-run training config


In [ ]:
!python train.py --config configs/colab_runtime_config.json --dry-run


## 9. Train or reuse a low-level checkpoint

Full training can take a long time. You may also set `CHECKPOINT_DIR` to a checkpoint from HW1.


In [ ]:
# Uncomment this cell to train a new low-level policy.
# !python train.py \
#   --config configs/colab_runtime_config.json \
#   --stage both \
#   --output-dir artifacts/low_level_train

CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "low_level_train" / "best_checkpoint"
print("checkpoint path:", CHECKPOINT_DIR)


## 10. Run starter track evaluation

This requires a real low-level checkpoint. If `CHECKPOINT_DIR` does not exist yet, this cell will skip safely. Once you have a checkpoint, it first runs a short no-render smoke test. The full video command remains commented so you can launch it intentionally.


In [ ]:
TRACK_EVAL_SMOKE_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval_smoke"
TRACK_EVAL_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval"

if CHECKPOINT_DIR.exists():
    print("Found checkpoint. Running short no-render track smoke eval...")
    !python run_track_bonus.py \
      --checkpoint-dir {CHECKPOINT_DIR} \
      --planner-config configs/starter_planner.json \
      --config configs/colab_runtime_config.json \
      --output-dir {TRACK_EVAL_SMOKE_DIR} \
      --duration-seconds 5 \
      --no-render
else:
    print("Skipping track eval: CHECKPOINT_DIR does not exist yet.")
    print("Train a policy above or set CHECKPOINT_DIR to your HW1 best_checkpoint, then rerun this cell.")

# Full evaluation with video, after the smoke run works:
# !python run_track_bonus.py \
#   --checkpoint-dir {CHECKPOINT_DIR} \
#   --planner-config configs/starter_planner.json \
#   --config configs/colab_runtime_config.json \
#   --output-dir {TRACK_EVAL_DIR} \
#   --render-every 10 \
#   --render-fps 5


## 11. Optional high-level planner search

This is a small black-box scaffold for tuning the JSON starter planner. It samples planner parameters, runs `run_track_bonus.py --no-render`, and keeps candidates with higher `scores.composite_score`. This is not analytic gradient descent through MuJoCo, and it is not expected to be the final method.


In [ ]:
HIGHLEVEL_DIR = COURSE_REPO_DIR / "artifacts" / "highlevel_train"

# !python train_highlevel_starter.py \
#   --checkpoint-dir {CHECKPOINT_DIR} \
#   --config configs/colab_runtime_config.json \
#   --output-dir {HIGHLEVEL_DIR} \
#   --iterations 8 \
#   --population 12

# After search, evaluate the best config:
# !python run_track_bonus.py \
#   --checkpoint-dir {CHECKPOINT_DIR} \
#   --planner-config {HIGHLEVEL_DIR / "best_planner_config.json"} \
#   --config configs/colab_runtime_config.json \
#   --output-dir artifacts/track_eval_best


## 12. Package submission


In [ ]:
import json
submission = {
    "team_name": "change_me",
    "checkpoint_dir": "best_checkpoint",
    "planner": "planner_config.json",
    "notes": "Briefly describe your low-level training and high-level planner."
}
(COURSE_REPO_DIR / "submission.json").write_text(json.dumps(submission, indent=2))
print((COURSE_REPO_DIR / "submission.json").read_text())


## 13. Final local checklist

Run this before submitting. It checks that the expected artifact paths exist; it does not judge policy quality.


In [ ]:
from pathlib import Path
expected = [
    COURSE_REPO_DIR / "submission.json",
    COURSE_REPO_DIR / "artifacts" / "track_eval" / "results.json",
]
for path in expected:
    print(path, "OK" if path.exists() else "MISSING")
print("Remember to include best_checkpoint/ and your planner_config.json or planner file in the submitted bundle.")
